# 07 Manual Review Analysis

Analyze the manually reviewed pilot after the review columns are filled.

In [14]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

PROCESSED = PROJECT_ROOT / "data" / "processed"
pilot_path = PROCESSED / "manual_review_pilot.csv"

df = pd.read_csv(pilot_path)
df.shape


(118, 26)

In [15]:
df.head()


,pilot_source,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,...,gt_reasoning,manual_review_status,manual_should_move,manual_primary_pin_type,manual_needs_multi_pin,manual_notes,tier,gt_model,offset_euclidean_m,offset_manhattan_m
0,high_offset,08f44f082e49c142032785383719bd54,Starke / Gainesville N.E. KOA Holiday,campground,FL,open_space,complex,high,0.9,41.354954,...,The pin is placed at the main access point nea...,ambiguous,true,vehicle_entry,True,"Open-space place; main vehicle access, gate, p...",3,gpt-4o-mini,41.401470,41.401470
1,high_offset,08f2661a16c4a6de0394c2b997482f2a,Subway,sandwich_shop,IN,standard_commercial,simple,low,0.9,37.810036,...,The pin is placed at the main customer entranc...,wrong_target,true,pedestrian_entry,False,High-offset standalone place; main customer en...,1,gpt-4o-mini,37.852565,45.902982
2,high_offset,08f489c00919806c0318e6399bbfe89e,Raising Cane's,fast_food_restaurant,TX,standard_commercial,simple,low,0.9,42.765474,...,The pin is placed at the main customer entranc...,wrong_target,true,pedestrian_entry,False,High-offset standalone place; main customer en...,1,gpt-4o-mini,42.813577,51.919092
3,high_offset,08f441ad446d54200323a1a9871a5d0d,Pinch A Penny Pool Patio Spa,hot_tubs_and_pools,FL,multi_tenant,simple,low,0.9,43.521618,...,The specific unit for 'Pinch A Penny Pool Pati...,ambiguous,true,pedestrian_entry,True,Multi-tenant or unit-level place; storefront c...,2,gpt-4o-mini,43.570571,52.837082
4,high_offset,08f275264c1acc110396d2360d5a93e9,Chipotle Mexican Grill,mexican_restaurant,MN,standard_commercial,simple,low,0.9,34.814880,...,The pin is placed at the main customer entranc...,wrong_target,true,pedestrian_entry,False,High-offset standalone place; main customer en...,1,gpt-4o-mini,34.854040,42.266736


In [16]:
manual_cols = [
    "manual_review_status",
    "manual_should_move",
    "manual_primary_pin_type",
    "manual_needs_multi_pin",
    "manual_notes",
]

df[manual_cols].isna().sum()


manual_review_status       0
manual_should_move         0
manual_primary_pin_type    0
manual_needs_multi_pin     0
manual_notes               0
dtype: int64

In [17]:
df["manual_review_status"].value_counts(dropna=False)


manual_review_status
privacy_sensitive    38
accepted             31
wrong_target         28
ambiguous            21
Name: count, dtype: int64

In [18]:
df["manual_should_move"].value_counts(dropna=False)


manual_should_move
false      69
true       41
unknown     8
Name: count, dtype: int64

In [19]:
df["manual_needs_multi_pin"].value_counts(dropna=False)


manual_needs_multi_pin
False    97
True     21
Name: count, dtype: int64

In [20]:
df["manual_primary_pin_type"].value_counts(dropna=False)


manual_primary_pin_type
current             69
pedestrian_entry    41
vehicle_entry        8
Name: count, dtype: int64

In [21]:
pd.crosstab(df["pilot_source"], df["manual_review_status"], dropna=False)


manual_review_status,accepted,ambiguous,privacy_sensitive,wrong_target
pilot_source,,,,
high_offset,0,12,0,28
low_confidence,0,0,30,0
multi_tenant,17,7,0,0
zero_offset_sample,14,2,8,0


In [22]:
pd.crosstab(df["tier_label"], df["manual_review_status"], dropna=False)


manual_review_status,accepted,ambiguous,privacy_sensitive,wrong_target
tier_label,,,,
multi_tenant,18,10,0,0
no_building,0,0,38,0
open_space,0,8,0,0
standard_commercial,13,3,0,28


In [23]:
pd.crosstab(df["pin_ambiguity"], df["manual_review_status"], dropna=False)


manual_review_status,accepted,ambiguous,privacy_sensitive,wrong_target
pin_ambiguity,,,,
high,1,3,38,1
low,30,15,0,27
medium,0,3,0,0


In [24]:
reviewed = df[df["manual_review_status"].notna() & (df["manual_review_status"] != "")].copy()

summary = {
    "reviewed_rows": len(reviewed),
    "accepted_rate_pct": round((reviewed["manual_review_status"] == "accepted").mean() * 100, 1) if len(reviewed) else 0,
    "wrong_target_rate_pct": round((reviewed["manual_review_status"] == "wrong_target").mean() * 100, 1) if len(reviewed) else 0,
    "ambiguous_rate_pct": round((reviewed["manual_review_status"] == "ambiguous").mean() * 100, 1) if len(reviewed) else 0,
    "multi_pin_needed_rate_pct": round((reviewed["manual_needs_multi_pin"].astype(str).str.lower() == "true").mean() * 100, 1) if len(reviewed) else 0,
}

summary


{'reviewed_rows': 118,
 'accepted_rate_pct': np.float64(26.3),
 'wrong_target_rate_pct': np.float64(23.7),
 'ambiguous_rate_pct': np.float64(17.8),
 'multi_pin_needed_rate_pct': np.float64(17.8)}

In [25]:
mismatch = reviewed[
    reviewed["should_move"].astype(str).str.lower()
    != reviewed["manual_should_move"].astype(str).str.lower()
].copy()

mismatch[
    [
        "pilot_source",
        "name",
        "category_primary",
        "tier_label",
        "pin_ambiguity",
        "offset_haversine_m",
        "should_move",
        "manual_should_move",
        "manual_review_status",
        "manual_notes",
    ]
].head(50)


,pilot_source,name,category_primary,tier_label,pin_ambiguity,offset_haversine_m,should_move,manual_should_move,manual_review_status,manual_notes
70,multi_tenant,United States Postal Service,shipping_center,multi_tenant,low,0.0,False,unknown,ambiguous,Multi-tenant or unit-level place; storefront c...
78,multi_tenant,HornerXpress Port St. Lucie,hot_tubs_and_pools,multi_tenant,low,0.0,False,unknown,ambiguous,Multi-tenant or unit-level place; storefront c...
80,multi_tenant,Redbox,rental_kiosks,multi_tenant,low,0.0,False,unknown,ambiguous,Multi-tenant or unit-level place; storefront c...
87,multi_tenant,Frontier's Little Academy,day_care_preschool,multi_tenant,low,0.0,False,unknown,ambiguous,Multi-tenant or unit-level place; storefront c...
89,multi_tenant,Redbox,rental_kiosks,multi_tenant,low,0.0,False,unknown,ambiguous,Multi-tenant or unit-level place; storefront c...
92,multi_tenant,Action Door & Trim Inc,building_supply_store,multi_tenant,low,0.0,False,unknown,ambiguous,Multi-tenant or unit-level place; storefront c...
94,zero_offset_sample,Gio's tacos,food_truck,open_space,low,0.0,False,unknown,ambiguous,"Open-space place; main vehicle access, gate, p..."
103,zero_offset_sample,Concept by Iowa Hearing by AudioNova,hearing_aids,standard_commercial,medium,0.0,False,unknown,ambiguous,Multi-tenant or unit-level place; storefront c...


In [26]:
summary_path = PROCESSED / "manual_review_analysis_summary.txt"

lines = [
    "Manual Review Analysis Summary",
    "",
    str(summary),
    "",
    "Review status counts",
    reviewed["manual_review_status"].value_counts(dropna=False).to_string(),
    "",
    "Manual primary pin type counts",
    reviewed["manual_primary_pin_type"].value_counts(dropna=False).to_string(),
    "",
    "Should-move mismatches",
    str(len(mismatch)),
]

summary_path.write_text("\n".join(lines) + "\n")
summary_path


WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed/manual_review_analysis_summary.txt')